## Quick demo KGraphPy

In [1]:
import cim_plugin   # Required to register the cimxml parser and serializer in the rdflib library
from rdflib import Literal, URIRef
from rdflib.namespace import RDF, DCAT, DCTERMS
from cim_plugin.utilities import load_graphs_from_cimxml, load_graphs_from_trig
from cim_plugin.header import CIMMetadataHeader
from pathlib import Path


### Import graph from file
- use load_graphs_from_cimxml for one or more cimxml file(s)
- use load_graphs_from_trig for one trig file

The sample files seen here can be downloaded from the [Nordic44 repo](https://github.com/statnett/Nordic44), along with many more.

In [2]:
input_file= Path.cwd().parents[1] / "Nordic44/instances/Grid/cimxml/Nordic44-HV_EQ.xml"

cims = load_graphs_from_cimxml([str(input_file)])

trig_file = Path.cwd().parents[1] / "Nordic44/instances/Grid/trig/Nordic44-HV_EQ.trig"

trigs = load_graphs_from_trig(str(trig_file))


### Handle the objects and identifiers
The load functions loads the graphs as a list of CIMProcessor objects. Each object can be extracted by indexing, and the graph identifier is stored as CIMProcessor.identifier.


In [3]:
g = cims[0]
t = trigs[0]

print("cimxml: ", g.identifier)
print("trig: ", t.identifier)

cimxml:  urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c
trig:  urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c


### Handling the graphs
The graph is stored in CIMProcess.graph. It can be handled with the usual rdflib functions and methods.

In [4]:
print("cimxml: ")
counter = 0
for s, p, o in g.graph:
    print(s, p, o)
    if isinstance(o, Literal):
        print(o.datatype)
    counter += 1
    if counter == 3:
        break

print()
print("trig: ")
counter = 0
for s, p, o in t.graph:
    print(s, p, o)
    if isinstance(o, Literal):
        print(o.datatype)
    counter += 1
    if counter == 3:
        break

cimxml: 
urn:uuid:f1769b8c-9aeb-11e5-91da-b8763fd99c5f http://iec.ch/TC57/CIM100#IdentifiedObject.description Branch 3359_8500_2  Limits <Default>
None
urn:uuid:f17696c9-9aeb-11e5-91da-b8763fd99c5f http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://iec.ch/TC57/CIM100#ConnectivityNode
urn:uuid:f1769c01-9aeb-11e5-91da-b8763fd99c5f http://iec.ch/TC57/CIM100#OperationalLimit.OperationalLimitSet urn:uuid:f1769bfe-9aeb-11e5-91da-b8763fd99c5f

trig: 
urn:uuid:8f2b8925-21b0-d14c-8ced-d67d3fcea0b7 http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://cim.ucaiug.io/ns#ConnectivityNode
urn:uuid:f1769ae3-9aeb-11e5-91da-b8763fd99c5f https://cim.ucaiug.io/ns#OperationalLimit.OperationalLimitType urn:uuid:f1769a40-9aeb-11e5-91da-b8763fd99c5f
urn:uuid:f1769d18-9aeb-11e5-91da-b8763fd99c5f https://cim.ucaiug.io/ns#IdentifiedObject.mRID f1769d18-9aeb-11e5-91da-b8763fd99c5f
None


### Header
CIMProcess.extract_header finds all the triples in the graph belonging to the metadata header, extracts them into another object and deletes them from the graph. The header can then be accessed through CIMProcess.graph.metadata_header.

In [5]:
g.extract_header()
t.extract_header()

print("cimxml: ")
for s, p, o in g.graph.metadata_header.triples:
    print(s, p, o)

print()
print("trig: ")
for s, p, o in t.graph.metadata_header.triples:
    print(s, p, o)

cimxml: 
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://iec.ch/TC57/61970-552/ModelDescription/1#FullModel
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c http://iec.ch/TC57/61970-552/ModelDescription/1#Model.created 2025-02-14T12:00:00Z
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c http://iec.ch/TC57/61970-552/ModelDescription/1#Model.modelingAuthoritySet http://www.Statnett.no/IGM/Nordic44_CGM
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c http://iec.ch/TC57/61970-552/ModelDescription/1#Model.scenarioTime 2014-12-31T23:00:00Z
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c http://iec.ch/TC57/61970-552/ModelDescription/1#Model.profile http://iec.ch/TC57/ns/CIM/CoreEquipment-EU/3.0
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398d8b59c http://iec.ch/TC57/61970-552/ModelDescription/1#Model.description Equipment (EQ) part of the Nordic 44-bus synthetic test model developed by Statnett SF of the Nordic region.
urn:uuid:e710212f-f6b2-8d4c-9dc0-365398

### Replace header
You can replace one header with a new header using CIMProcess.replace_header. You can build the new header up triple by triple, or collect it from another file

In [ ]:
new_header = CIMMetadataHeader.empty(URIRef("header_example")) # Creating a new empty header
new_header.graph.bind("ex", "example.org")  # Binding a new namespace
new_header.add_triple(RDF.type, DCAT.Dataset)   # The rdf.type triple is important to recognise the header
new_header.add_triple(DCTERMS.conformsTo, URIRef("http://iec.ch/TC57/ns/CIM/CoreEquipment-EU/3.0"))
new_header.add_triple(URIRef("example.org/predicate"), Literal("example object"))   # Adding a triple
t.replace_header(new_header)

print("trig: ")
for s, p, o in t.graph.metadata_header.triples:
    print(s, p, o)

trig: 
header_example http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/ns/dcat#Dataset
header_example example.org/predicate example object
header_example http://purl.org/dc/terms/conformsTo http://iec.ch/TC57/ns/CIM/CoreEquipment-EU/3.0


### Validate header
You can check if the header follows the cim standard for dcat:Dataset headers. The validation will correct where automatic correction is possible. Logs are emitted for every correction as well as for errors that must be fixed manually.
Trig and cimxml has slightly different standards, so format must be given. Default is "cimxml". 

In [8]:
g.validate_header()
t.validate_header(format="trig")

Validation for MD.FullModel header is not implemented yet. No validation performed for this header type.
Missing required dcterms:issued triple.
Missing required rdf:type rdfg:Graph triple for Trig header. Adding it.


### Namespaces
Namespaces can be changed using CIMProcess.update_namespace. Though rdflibs graph.bind allows binding of new namespaces, it does not update the triples. This method does both, but can only be used on namespaces that already exists.

In [9]:
t.update_namespace("cim", g.graph.namespace_manager.store.namespace("cim")) # Updating the trig graph with the cim namespace from the cimxml graph.

print("trig: ")
counter = 0
for s, p, o in t.graph:
    print(s, p, o)
    counter += 1
    if counter == 3:
        break

trig: 
urn:uuid:f1769b8c-9aeb-11e5-91da-b8763fd99c5f http://iec.ch/TC57/CIM100#IdentifiedObject.description Branch 3359_8500_2  Limits <Default>
urn:uuid:f1769e04-9aeb-11e5-91da-b8763fd99c5f http://iec.ch/TC57/CIM100#TapChanger.ltcFlag true
urn:uuid:2dd90468-bdfb-11e5-94fa-c8f73332c8f4 http://iec.ch/TC57/CIM100#PowerTransformerEnd.x 2.646


### Namespace validation
Namespaces can be checked whether they are correct according to the CIM standard. This will also fix them if they are wrong.

If you plan to output as cimxml you should use the cimxml_format=True option. It checks whether the graph profile belongs to the CGMES exceptions and will therefore check the prefixes "cim", "eu" and "md" against the CGMES standard. 

In [10]:
t.validate_namespaces(cimxml_format=True)
g.validate_namespaces()

Wrong namespace detected for 'eu': 'https://cim.ucaiug.io/ns/eu#'. Namespace corrected to 'http://iec.ch/TC57/CIM100-EuropeanExtension/1/0#'.
Wrong namespace detected for 'cim': 'http://iec.ch/TC57/CIM100#'. Namespace corrected to 'https://cim.ucaiug.io/ns#'.
Wrong namespace detected for 'eu': 'http://iec.ch/TC57/CIM100-EuropeanExtension/1/0#'. Namespace corrected to 'https://cim.ucaiug.io/ns/eu#'.


### Serialize to cimxml
To serialize to file the CIMProcessor.to_file method can be used. For cimxml, the header must be seperate from the graph (into CIMProcessor.graph.metadata_header). If it is not, the header will be serialized as regular triples. The file will then not be a standard cimxml file.
The qualifier decides how uuids will be written in the data. The options are "underscore", "urn" and "namespace".

Tip! If you get a lot of errors saying profile cannot be found, it is because t.graph.metadata_header.profile is None. The presence of a triple with DCTERMS.conformsTO or MD.Model.profile predicate is required in the header to find the profile. The profile decides whether subjects are serialized with rdf:ID or rdf:about, with rdf:about as default when no profile match is found.

In [ ]:
output_file_cim = Path.cwd().parent / "fromtrig_tocimxml_grid_eq.xml"
t.to_file(output_file_cim, format="cimxml", qualifier="underscore")


### Serialize to trig
When serializing to trig, datatypes can be added to literals. That requires a linkml file with the datatype information. 
The enriching can be done before serializing, allowing for namespaces being a little different, or it can be done in the serializing step itself.

The sample linkml file used here can be downloaded from the [PowerSystemPy repo](https://github.com/statnett/PowerSystemPy).

In [18]:
linkmlfile = Path.cwd().parent / "CoreEquipment.linkml.yaml"
g.set_schema(linkmlfile)
g.enrich_literal_datatypes(allow_different_namespaces=True) # You can allow the namespaces in the graph to differ from the model, but the prefix must then be the same

output_file_trig = Path.cwd().parent / "fromcimxml_totrig_grid_eq.trig"
g.to_file(output_file_trig, format="trig") 
# You can do the enriching directly in the serialization step, but allow_different_namespaces is not supported, so namespaces must match:
# g.to_file(output_file_trig, format="trig", schema_path=linkmlfile, enrich_datatypes=True)


dc namespace is already mapped to http://purl.org/dc/terms/ - Overriding with mapping to http://purl.org/dc/elements/1.1/
